# Shipment Tracking Pipeline
A synthetic data engineering exercise. Run cells in order. Results are not pre-executed. See README.md for the data contract and limitations.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = ""  # Optional: enter a writable catalog name.
RESET_DEMO = True  # Resets only this dedicated demonstration's tables.
if CATALOG:
    spark.sql("USE CATALOG `" + CATALOG.replace("`", "``") + "`")
catalog = spark.sql("SELECT current_catalog()").first()[0]
spark.sql("CREATE SCHEMA IF NOT EXISTS shipment_portfolio_demo")
spark.sql("USE SCHEMA shipment_portfolio_demo")
spark.sql("CREATE VOLUME IF NOT EXISTS landing")
landing_path = f"/Volumes/{catalog}/shipment_portfolio_demo/landing"
if RESET_DEMO:
    for table in ["bronze_shipments", "rejected_shipments", "silver_shipments", "gold_shipment_summary"]:
        if spark.catalog.tableExists(table):
            spark.sql(f"TRUNCATE TABLE {table}")
print(landing_path)


## Create synthetic incoming files
The repository CSVs are embedded below so importing this notebook is sufficient. In a real project, a source system delivers these files.

In [ ]:
dbutils.fs.put(f"{landing_path}/morning.csv", 'shipment_id,city,status,version\nS101,Dallas,SHIPPED,1\nS102,Austin,PROCESSING,1\nS103,Houston,DELIVERED,1\nS103,Houston,DELIVERED,1\nS104,Plano,UNKNOWN,1\n', overwrite=True)
dbutils.fs.put(f"{landing_path}/afternoon.csv", 'shipment_id,city,status,version\nS101,Dallas,DELIVERED,2\nS104,Plano,SHIPPED,2\n', overwrite=True)

## Transform, validate, and merge
Bronze preserves input strings. Quarantine retains the first validation failure. Silver retains the highest valid version per ID. Exact replay is safe for Silver; audit tables append on every attempt.

In [ ]:
def process_file(filename):
    raw = (spark.read.option("header", True)
           .schema("shipment_id STRING, city STRING, status STRING, version STRING")
           .csv(f"{landing_path}/{filename}")
           .withColumn("source_file", F.lit(filename))
           .withColumn("ingested_at", F.current_timestamp()))
    raw.write.format("delta").mode("append").saveAsTable("bronze_shipments")
    cleaned = (raw.withColumn("shipment_id", F.trim("shipment_id"))
               .withColumn("city", F.trim("city"))
               .withColumn("status", F.upper(F.trim("status")))
               .withColumn("version", F.expr("try_cast(version AS INT)")))
    checked = cleaned.withColumn("rejection_reason",
        F.when(F.col("shipment_id").isNull() | (F.col("shipment_id") == ""), "Missing shipment ID")
         .when(F.col("city").isNull() | (F.col("city") == ""), "Missing city")
         .when(F.col("status").isNull() | ~F.col("status").isin("PROCESSING", "SHIPPED", "DELIVERED"), "Invalid shipment status")
         .when(F.col("version").isNull() | (F.col("version") <= 0), "Invalid version"))
    rejected = checked.filter(F.col("rejection_reason").isNotNull())
    rejected.write.format("delta").mode("append").saveAsTable("rejected_shipments")
    valid = (checked.filter(F.col("rejection_reason").isNull())
             .select("shipment_id", "city", "status", "version").dropDuplicates())
    conflicts = valid.groupBy("shipment_id", "version").count().filter(F.col("count") > 1)
    if conflicts.limit(1).count():
        raise ValueError("Conflicting values for the same shipment and version")
    ranking = Window.partitionBy("shipment_id").orderBy(F.col("version").desc())
    latest = (valid.withColumn("rn", F.row_number().over(ranking))
              .filter(F.col("rn") == 1).drop("rn"))
    latest.createOrReplaceTempView("incoming_shipments")
    spark.sql("""CREATE TABLE IF NOT EXISTS silver_shipments (
        shipment_id STRING, city STRING, status STRING, version INT
    ) USING DELTA""")
    spark.sql("""MERGE INTO silver_shipments AS target
        USING incoming_shipments AS source
        ON target.shipment_id = source.shipment_id
        WHEN MATCHED AND source.version > target.version THEN UPDATE SET
            target.city = source.city, target.status = source.status,
            target.version = source.version
        WHEN NOT MATCHED THEN INSERT (shipment_id, city, status, version)
            VALUES (source.shipment_id, source.city, source.status, source.version)
    """)
    print(f"Processed: {filename}")


## Process morning and verify
Five input rows produce three valid unique shipments and one rejected row.

In [ ]:
process_file("morning.csv")
assert spark.table("silver_shipments").count() == 3
assert spark.table("bronze_shipments").count() == 5
assert spark.table("rejected_shipments").count() == 1
display(spark.table("silver_shipments").orderBy("shipment_id"))
display(spark.table("rejected_shipments"))

## Process afternoon and verify
S101 updates to DELIVERED. S104 is now valid and is inserted.

In [ ]:
process_file("afternoon.csv")
expected = [
    ("S101", "Dallas", "DELIVERED", 2),
    ("S102", "Austin", "PROCESSING", 1),
    ("S103", "Houston", "DELIVERED", 1),
    ("S104", "Plano", "SHIPPED", 2),
]
def silver_snapshot():
    # collect() is appropriate only for this tiny synthetic test fixture.
    return [tuple(r) for r in spark.table("silver_shipments").orderBy("shipment_id").collect()]
assert silver_snapshot() == expected
assert spark.table("bronze_shipments").count() == 7
display(spark.table("silver_shipments").orderBy("shipment_id"))

## Check reruns, out-of-order input, and conflicts
These checks add processing attempts to Bronze/quarantine. They must not change the latest valid Silver state.

In [ ]:
process_file("afternoon.csv")
assert silver_snapshot() == expected
process_file("morning.csv")
assert silver_snapshot() == expected
conflict_csv = "shipment_id,city,status,version\nS101,Dallas,SHIPPED,3\nS101,Dallas,DELIVERED,3\n"
dbutils.fs.put(f"{landing_path}/conflict.csv", conflict_csv, overwrite=True)
try:
    process_file("conflict.csv")
except ValueError as error:
    assert "Conflicting values" in str(error)
else:
    raise AssertionError("Expected same-version conflict to fail")
assert silver_snapshot() == expected
print("Passed: replay, older input, and conflict checks")

## Refresh Gold and validate business totals
This report counts shipments, not incoming event rows.

In [ ]:
spark.sql("""CREATE OR REPLACE TABLE gold_shipment_summary USING DELTA AS
    SELECT status, COUNT(*) AS shipment_count FROM silver_shipments GROUP BY status
""")
counts = {r["status"]: r["shipment_count"] for r in spark.table("gold_shipment_summary").collect()}
assert counts == {"DELIVERED": 2, "PROCESSING": 1, "SHIPPED": 1}
display(spark.table("gold_shipment_summary").orderBy("status"))
print("All demonstration assertions passed. Record this run in docs/PROGRESS.md.")

## Next steps
Record actual results and screenshots, commit them to GitHub, then add file discovery, ingestion tracking, scheduling, and alerts. This version is a sequential learning project, not a deployed service.